In [ ]:
from __future__ import annotations

import warnings
import h5py as h5
import numpy as np
from typing import Sequence
from compas_python_utils.cosmic_integration.ClassCOMPAS import COMPASData


def process_to_h5(
    path_to_data: str,
    outfile: str = "compas_compact.h5",
    reload: bool = False,
    verbose: bool = False,
) -> str:
    """
    Convert a COMPAS output HDF5 to a compact training file by:
      - Copying groups: BSE_System_Parameters, BSE_Double_Compact_Objects, Run_details
      - Creating BSE_DCO_Masks/{DCO_Mask, BBH_Mask} of shape (n_systems,)

    Returns:
      outfile path on success, "" if BBH backend fails or no DCOs formed.
    """
    # --- Read source, compute masks ---------------------------------------------------------
    with h5.File(path_to_data, "r") as src:
        # Required groups
        if "BSE_System_Parameters" not in src:
            raise KeyError("Missing 'BSE_System_Parameters' in input HDF5.")

        sp = src["BSE_System_Parameters"]
        if "SEED" not in sp:
            raise KeyError("Missing 'SEED' in 'BSE_System_Parameters'.")

        seeds = np.asarray(sp["SEED"])
        if seeds.ndim != 1:
            raise ValueError(f"'SEED' must be 1D, got shape {seeds.shape}.")

        # DCO mask
        if "BSE_Double_Compact_Objects" not in src:
            if verbose:
                print("No DCOs formed.")
            warnings.warn("No DCOs formed; cannot build masks.")
            return ""

        dco_seeds = np.asarray(src["BSE_Double_Compact_Objects"]["SEED"])
        unique_dco = np.unique(dco_seeds)
        dco_mask = np.isin(seeds, unique_dco).astype(np.int8)

        # sanity check
        if int(dco_mask.sum()) != int(len(unique_dco)):
            raise AssertionError(
                "Expected number of DCOs from simulation did not equal inferred number."
            )
        if verbose:
            print(f"DCOs unique seeds: {len(unique_dco)}; mask sum: {int(dco_mask.sum())}")

        # BBH mask (within Hubble time, pessimistic)
        try:
            bbhs = COMPASData(path_to_data)
            bbhs.setCOMPASDCOmask(types="BBH", withinHubbleTime=True, pessimistic=True)
            bbhs.setCOMPASData()
            bbh_seeds = np.asarray(bbhs.seedsDCO)
            bbh_mask = np.isin(seeds, bbh_seeds).astype(np.int8)

            if int(bbh_mask.sum()) > int(dco_mask.sum()):
                raise AssertionError("BBH_Mask sum should be ≤ DCO_Mask sum.")
            if verbose:
                print(f"BBH seeds: {len(np.unique(bbh_seeds))}; mask sum: {int(bbh_mask.sum())}")
        except Exception as e:
            warnings.warn(f"BBH backend issue: {e}. Will not write this file.")
            return ""

        # --- Write destination file ---------------------------------------------------------
        with h5.File(outfile, "w") as dst:
            def _copy_group_if_present(name: str):
                if name in src:
                    # h5py copy does I/O without loading full datasets into RAM
                    src.copy(name, dst, name=name)
                    if verbose:
                        print(f"Copied group '{name}'")
                else:
                    if verbose:
                        print(f"Group '{name}' not found; skipping.")

            # Copy requested groups verbatim
            _copy_group_if_present("BSE_System_Parameters")
            _copy_group_if_present("BSE_Double_Compact_Objects")
            _copy_group_if_present("Run_details")

            # Add masks
            masks_grp = dst.require_group("BSE_DCO_Masks")
            masks_grp.create_dataset("DCO_Mask", data=dco_mask, dtype=np.int8)
            masks_grp.create_dataset("BBH_Mask", data=bbh_mask, dtype=np.int8)

    if reload:
        with h5.File(outfile, "r") as r:
            if verbose:
                print("--- h5 file reload ---")
                print(list(r.keys()))
                print("BSE_DCO_Masks keys:", list(r["BSE_DCO_Masks"].keys()))
                print("DCO_Mask shape:", r["BSE_DCO_Masks"]["DCO_Mask"].shape)
                print("BBH_Mask shape:", r["BSE_DCO_Masks"]["BBH_Mask"].shape)

    return outfile